# Capítulo 4 — Avaliando sistemas de IA

**Livro:** começa na página **304** do PDF.

📗 **Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Capacidade de seguir instruções · PDF p. 326

#### 🧪 Exercício: Checar instruções automaticamente

Peça ao modelo: *"Liste 3 vantagens do RAG, em JSON com a chave 'vantagens', cada item com no máximo 8 palavras"*. Escreva verificações em Python: o JSON é válido? tem 3 itens? cada item tem ≤ 8 palavras?

In [ ]:
import json
saida = perguntar("Liste 3 vantagens do RAG em JSON com a chave 'vantagens' (lista de textos), "
                  "cada item com no máximo 8 palavras.", temperature=0, response_format={"type": "json_object"})
checagens = {}
try:
    dados = json.loads(saida); checagens["json válido"] = True
except json.JSONDecodeError:
    dados = {}; checagens["json válido"] = False
itens = dados.get("vantagens", [])
checagens["3 itens"] = len(itens) == 3
checagens["≤ 8 palavras"] = all(len(i.split()) <= 8 for i in itens)
print(itens); checagens

### Custo e latência · PDF p. 335

#### 🧪 Exercício: Medir tempo até o 1º token e tempo total

Use `cliente.chat.completions.create(..., stream=True)` para medir o **tempo até o primeiro token (TTFT)** e o **tempo total**. Depois veja o custo com `gasto()`.

In [ ]:
import time
inicio = time.perf_counter(); primeiro = None; pedacos = []
fluxo = cliente.chat.completions.create(model=MODELO, stream=True,
            messages=[{"role": "user", "content": "Explique o que é RAG em 3 frases."}])
for parte in fluxo:
    if parte.choices and parte.choices[0].delta.content:
        primeiro = primeiro or time.perf_counter()
        pedacos.append(parte.choices[0].delta.content)
fim = time.perf_counter()
print("".join(pedacos))
print(f"TTFT: {primeiro - inicio:.2f}s · total: {fim - inicio:.2f}s")

### Etapa 3. Definir métodos de avaliação e dados · PDF p. 386

#### 🧪 Exercício: Seu primeiro conjunto de avaliação

Monte 8 exemplos (texto → categoria esperada) para o classificador do cap. 1. Rode o classificador em todos e calcule a **acurácia**. Mostre os erros numa tabela. Esse é o seu *eval set* (p. 386–393).

In [ ]:
import pandas as pd
exemplos = [("A entrega atrasou 5 dias", "reclamação"), ("Muito obrigado pela ajuda!", "elogio"),
            ("Vocês aceitam boleto?", "dúvida"), ("O produto veio quebrado", "reclamação"),
            ("Equipe nota 10", "elogio"), ("Qual o horário de atendimento?", "dúvida"),
            ("Bom dia", "outro"), ("Cobraram duas vezes no cartão", "reclamação")]
sistema = ("Classifique a mensagem em exatamente uma categoria: reclamação, elogio, dúvida ou outro. "
           "Responda só a categoria, em minúsculas.")
df = pd.DataFrame(exemplos, columns=["texto", "esperado"])
df["previsto"] = [perguntar(t, sistema=sistema, temperature=0).strip(" .") for t in df.texto]
print("Acurácia:", (df.previsto == df.esperado).mean())
df[df.previsto != df.esperado]

## 💰 Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou